# Session 10: OOP Principles and SOLID Fundamentals

**Python Fundamentals | Session 10 of 14 | Guided learning: 30 minutes**

In Session 9, the Learning Assistant represented study activities as objects. Now we will protect object state, reuse behaviour and separate responsibilities. Follow **Concept → Predict → Run → Modify → Verify → Reflect**. This notebook uses Python's built-in features only and runs in a fresh Google Colab runtime.

## 1. Why OOP Design Matters (2 minutes)

**CONCEPT:** A basic class works, but public attributes allow invalid state:

```python
session = StudySession("Python", 45)
session.duration = -100
```

**PREDICT:** Should a study session allow a negative duration? No. A negative duration cannot represent real study time, so the object should protect its state.

Four useful design tools are **encapsulation** (protect state), **inheritance** (reuse a parent type), **polymorphism** (one operation with suitable implementations) and **abstraction** (focus on essential behaviour). They are tools, not rules that every programme must use.

In [ ]:
class BasicStudySession:
    def __init__(self, topic, duration):
        self.topic = topic
        self.duration = duration

session = BasicStudySession("Python", 45)
session.duration = -100
print(session.topic, session.duration)

**MODIFY 1:** Explain why the assignment above should be rejected. What other invalid duration values can you think of?

**REFLECT:** How does protecting state make later calculations more trustworthy?

## 2. Encapsulation — Protecting Object State (4 minutes)

**CONCEPT:** A `property` lets a class control how an attribute is read and changed. `_duration` is an internal attribute by convention; a leading underscore does not make it completely inaccessible. The setter checks every initial value and later update. `ValueError` is a signal that a supplied value is unacceptable.

**PREDICT:** Which values should be accepted: `45`, `0`, `-5`, `30.5` and `True`? Only positive integers should be accepted. In Python, `bool` is a subclass of `int`, so the exact `type(value) is int` check deliberately rejects `True`.

In [ ]:
class StudySession:
    def __init__(self, topic, duration):
        self.topic = topic
        self.duration = duration

    @property
    def duration(self):
        return self._duration

    @duration.setter
    def duration(self, value):
        if type(value) is not int or value <= 0:
            raise ValueError("Duration must be a positive integer")
        self._duration = value

    def to_hours(self):
        return round(self.duration / 60, 2)

    def describe(self):
        return f"Individual study: {self.topic} ({self.duration} min)"

valid_session = StudySession("Python", 45)
valid_session.duration = 60
print(valid_session.duration, valid_session.to_hours())
assert valid_session.duration == 60
assert valid_session.to_hours() == 1.0

In [ ]:
def assignment_is_rejected(session, value):
    previous_duration = session.duration
    try:
        session.duration = value
    except ValueError:
        return session.duration == previous_duration
    return False

for invalid_value in (0, -5, 30.5, True):
    assert assignment_is_rejected(valid_session, invalid_value)
assert valid_session.duration == 60
print("Invalid values were rejected and the valid state was preserved.")

**MODIFY 2:** Try a valid positive integer such as `90`, then try an invalid value and check that the previous duration remains unchanged.

**VERIFY:** The property controls both construction and assignment. `_duration` can technically be accessed, but the underscore tells other programmers to use the validated `duration` property.

**REFLECT:** Does a leading underscore make an attribute completely inaccessible? No; it is a convention for intended internal use.

## 3. Inheritance — Reusing Existing Behaviour (4 minutes)

**CONCEPT:** A guided study session has all the data and behaviour of a normal session, plus a mentor. `GuidedStudySession` is the subclass and `StudySession` is its parent. `super().__init__()` reuses the parent's initialisation, including duration validation.

**PREDICT:** Will the guided object have `topic`, `duration`, `mentor` and inherited `to_hours()`?

In [ ]:
class GuidedStudySession(StudySession):
    def __init__(self, topic, duration, mentor):
        super().__init__(topic, duration)
        self.mentor = mentor

guided = GuidedStudySession("Mathematics", 60, "Alex")
print(guided.topic)
print(guided.duration)
print(guided.mentor)
print(guided.to_hours())
assert guided.topic == "Mathematics"
assert guided.duration == 60
assert guided.to_hours() == 1.0


**MODIFY 3:** Create `another_guided = GuidedStudySession("English", 25, "Sam")`. Why is calling `super().__init__()` better than duplicating the parent's validation in the subclass?

**Expected result:** Reusing one validation rule prevents the parent and subclass from becoming inconsistent when requirements change.

## 4. Method Overriding and Polymorphism (4 minutes)

**CONCEPT:** A subclass can override a method to provide behaviour suited to its extra information. Polymorphism means different object types respond to the same method call in their own way.

**PREDICT:** What will the mixed loop print? We do not need an `if` checking each type; Python selects the appropriate `describe()` implementation.

In [ ]:
class GuidedStudySession(StudySession):
    def __init__(self, topic, duration, mentor):
        super().__init__(topic, duration)
        self.mentor = mentor

    def describe(self):
        return (
            f"Guided study: {self.topic} "
            f"({self.duration} min) with {self.mentor}"
        )

sessions = [
    StudySession("Python", 45),
    GuidedStudySession("Mathematics", 60, "Alex"),
    StudySession("English", 25),
]

descriptions = []
for session in sessions:
    descriptions.append(session.describe())
    print(session.describe())

assert descriptions[0] == "Individual study: Python (45 min)"
assert descriptions[1] == "Guided study: Mathematics (60 min) with Alex"
assert descriptions[2] == "Individual study: English (25 min)"

**MODIFY 4:** Change Alex's name to `"Jordan"` and rerun the loop. The guided description should change while the individual descriptions remain the same.

**REFLECT:** Why can the loop call `describe()` without checking whether each object is guided? Each object supplies the method implementation appropriate to its type.

## 5. Abstraction — Using Behaviour Without Knowing Every Detail (3 minutes)

**CONCEPT:** Abstraction means focusing on what an object can do rather than how every detail is implemented. Reporting code needs descriptions; it does not need to know how an individual or guided description is assembled. Abstraction and polymorphism work together here.

**PREDICT:** What information does this reporting code really need? Only the string returned by `describe()`.

In [ ]:
def display_descriptions(items):
    return [item.describe() for item in items]

report_lines = display_descriptions(sessions)
print("\n".join(report_lines))
assert len(report_lines) == 3
assert report_lines == descriptions

**MODIFY 5:** Add another object type only if it can provide a compatible `describe()` method. The reporting function should not need new type checks.

**REFLECT:** What would be harder to maintain if the report contained separate formatting branches for every session type?

## 6. Single Responsibility Principle (4 minutes)

**CONCEPT:** **S — Single Responsibility Principle (SRP)** says that a class should have one clear responsibility or reason to change. An overloaded `StudySession` that stores data, reads input, calculates every statistic and prints the whole report would be harder to test and maintain.

A clearer separation is: `StudySession` validates one activity; `GuidedStudySession` adds mentor data; `StudyTracker` manages collections and totals; `display_descriptions()` formats output.

**PREDICT:** Which component should validate a duration? The session object, because validity is part of its own state.

In [ ]:
class StudyTracker:
    def __init__(self):
        self.sessions = []

    def add_session(self, session):
        self.sessions.append(session)

    def total_minutes(self):
        return sum([session.duration for session in self.sessions])

    def get_descriptions(self):
        return [session.describe() for session in self.sessions]

tracker = StudyTracker()
tracker.add_session(StudySession("Python", 45))
tracker.add_session(GuidedStudySession("Mathematics", 60, "Alex"))
print(tracker.total_minutes())
print(tracker.get_descriptions())
assert tracker.total_minutes() == 105
assert len(tracker.get_descriptions()) == 2

In [ ]:
responsibilities = {
    "validate duration": "StudySession",
    "add a session to a collection": "StudyTracker",
    "print a report heading": "display_descriptions",
    "calculate total study time": "StudyTracker",
}
assert responsibilities["validate duration"] == "StudySession"
assert responsibilities["add a session to a collection"] == "StudyTracker"
assert responsibilities["print a report heading"] == "display_descriptions"
print("SRP responsibility matches:", responsibilities)

**MODIFY 6:** Add a responsibility to the matching dictionary, such as `"store mentor information": "GuidedStudySession"`. Keep each responsibility assigned to the smallest suitable component.

**REFLECT:** What problems might appear when one class performs input, validation, calculations and presentation? Unrelated changes become harder to isolate and test.

## 7. SOLID — Five Principles at a Glance (3 minutes)

| Principle | Beginner-friendly meaning |
| --- | --- |
| **S — Single Responsibility** | Give each component a clear responsibility. |
| **O — Open/Closed** | Prefer extending behaviour without repeatedly changing working code. |
| **L — Liskov Substitution** | A suitable subclass should work where its parent is expected. |
| **I — Interface Segregation** | Avoid forcing components to depend on behaviour they do not need. |
| **D — Dependency Inversion** | Higher-level logic should depend on suitable abstractions rather than unnecessary implementation details. |

**CONCEPT:** We are not implementing formal interfaces or dependency-injection frameworks today. SRP is visible in our separation; Open/Closed is illustrated by adding `GuidedStudySession`; Liskov means the guided object still works as a study session. Interface Segregation and Dependency Inversion remain useful design ideas for larger programmes.

**PREDICT:** Which principle best matches adding a new session subtype without rewriting the reporting loop?

In [ ]:
solid_matches = {
    "one clear reason to change": "S",
    "add guided behaviour while keeping the report loop": "O",
    "guided session works as a study session": "L",
}
assert solid_matches["one clear reason to change"] == "S"
assert solid_matches["add guided behaviour while keeping the report loop"] == "O"
assert solid_matches["guided session works as a study session"] == "L"
print("SOLID matching:", solid_matches)

**EXPERIMENT:** Match these scenarios yourself before checking the dictionary: avoid forcing a client to use unused methods (**I**), and let high-level reporting use a common session behaviour rather than a concrete formatting detail (**D**).

**REFLECT:** Why is it better to learn what each principle helps us notice rather than memorise five labels without examples?

## 8. Debug AI-Generated OOP Code (4 minutes)

**DEBUG 1 — Missing parent initialisation:**
```python
class GuidedStudySession(StudySession):
    def __init__(self, topic, duration, mentor):
        self.mentor = mentor
```
Use `super().__init__(topic, duration)` so inherited attributes and validation are set.

**DEBUG 2 — Shared mutable class attribute:**
```python
class StudyTracker:
    sessions = []
```
Create `self.sessions = []` in `__init__()` so trackers are independent.

**DEBUG 3 — Unnecessary type checks:**
```python
if type(session) == GuidedStudySession:
    ...
else:
    ...
```
Call the shared `session.describe()` method instead.

**DEBUG 4 — Duplicated responsibilities:** A tracker that directly prints a complete report mixes collection management and presentation. Return data or descriptions, then let a reporting function format them.

These faulty examples remain in Markdown so Run all never executes intentionally broken code. **PREDICT:** Which assertions would distinguish the corrected design?

In [ ]:
corrected_guided = GuidedStudySession("Mathematics", 60, "Alex")
tracker_a = StudyTracker()
tracker_b = StudyTracker()
tracker_a.add_session(StudySession("Python", 45))
tracker_a.add_session(corrected_guided)
tracker_a.add_session(StudySession("Python", 30))

assert corrected_guided.topic == "Mathematics"
assert corrected_guided.to_hours() == 1.0
assert tracker_a.sessions is not tracker_b.sessions
assert tracker_b.total_minutes() == 0
assert len(tracker_a.get_descriptions()) == 3
assert tracker_a.total_minutes() == 135
assert tracker_a.get_descriptions()[0] == "Individual study: Python (45 min)"
assert tracker_a.get_descriptions()[1] == "Guided study: Mathematics (60 min) with Alex"
assert tracker_a.get_descriptions()[2] == "Individual study: Python (30 min)"
print("Corrected design passes behaviour checks.")

**VERIFY:** Passing tests verifies behaviour. Code inspection is still needed to verify design choices such as using `super()`, independent instance lists and polymorphic method calls.

**MODIFY:** Add another repeated Python session and check that it remains a separate object in the tracker.

## 9. Integration Challenge and Reflection (2 minutes)

**CONCEPT:** The final tracker combines individual and guided sessions without changing the reporting code.

**PREDICT:** What total and descriptions will this tracker produce?

In [ ]:
final_tracker = StudyTracker()
final_tracker.add_session(StudySession("Python", 45))
final_tracker.add_session(GuidedStudySession("Mathematics", 60, "Alex"))
final_tracker.add_session(StudySession("Python", 30))

assert final_tracker.total_minutes() == 135
assert final_tracker.get_descriptions() == [
    "Individual study: Python (45 min)",
    "Guided study: Mathematics (60 min) with Alex",
    "Individual study: Python (30 min)",
]
independent_tracker = StudyTracker()
assert independent_tracker.total_minutes() == 0
assert independent_tracker.get_descriptions() == []
print("Final integration checks passed.")

**REFLECT:**

1. What problem does encapsulation solve? It protects object state by validating changes.
2. Why do we use `super().__init__()`? It reuses the parent's initialisation and avoids duplicated rules.
3. How does polymorphism simplify our code? One method call works with several suitable object types.
4. What does SRP encourage us to do? Give each component one clear responsibility.
5. Why verify both behaviour and design after AI-generated changes? Passing outputs is not enough if the code creates shared state or unnecessary coupling.

**VERIFY:** The notebook uses a validated property, inheritance, two polymorphism demonstrations, independent trackers, SRP and SOLID matching activities, and final assertions.

**Good OOP design is not about creating more classes. It is about protecting data, reusing behaviour, separating responsibilities and making software easier to extend and verify.**